# Outings from the usual workplace: mode (walk against the mode of the work tour, binary logit)

Re-estimate the model with Biogeme and check it against the article (est_subtour.tex, third table (tab:est_subtour_mode)).

This notebook goes from the raw data to the estimates of this model:

1. Raw data
2. Data preparation
3. Estimation data
4. Descriptive statistics
5. Correlation of the explanatory variables (heatmaps, VIF)
6. Model specification (the folder's `model/estimate_outing_mode.py`, unchanged)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

**Kernel:** select *Python (biogeme)* (`~/.venvs/biogeme`) in VS Code, then *Run All*. The switches in the next cell decide what is recomputed.

In [ ]:
RUN_DATA_PREP  = False   # True: rebuild the estimation tables (section 2)
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = True    # Biogeme estimation, written to output/
RUN_HOLDOUT    = True    # log-likelihood of the hold-out half at the full-precision estimates

import os, sys, json, glob, inspect, importlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "outing_mode"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", FOLDER)

## 1. Raw data

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**2017/18 Regional Travel Survey (Baltimore and Washington regions), household travel survey, weekday travel day** — Baltimore Metropolitan Council (BMC) and the National Capital Region Transportation Planning Board (TPB) at the Metropolitan Washington Council of Governments (MWCOG)
- Where: <https://www.baltometro.org/ and https://www.mwcog.org/ (verify: no download page is documented in the project; the data are released on request)>
- How: Request the 2017/18 Regional Travel Survey public-use files from BMC (Baltimore region) or MWCOG/TPB (Washington region). The project used one flat file in which every trip row carries its household and person fields (household, person, trip and vehicle tables joined): RTS_raw_flat_data.csv, with the codebook RTS_codebook_data_dictionary.xlsx (sheets Household, Person, Trip, Vehicle). The two cleaned files rts_trips_clean.csv (163,290 trips, deduplicated) and rts_tours.csv (57,627 home-based tours) were made by 01_clean_rts.py of the tour-based MITO project; that script is not on this machine, so these two files are shipped and cannot be rebuilt here. Survey weights: wthhfin, wtperfin, wttrdfin. If you receive separate household / person / trip / vehicle tables instead, join them on household_id, person_id (persno) and tripid to the flat layout of the codebook.
- Put in: `00_shared_data/raw/rts_2017_18 (the project's copies are provided in 00_shared_data/rts_2017_18/raw as .csv.gz)`
- Files: `RTS_raw_flat_data.csv`, `RTS_codebook_data_dictionary.xlsx`, `rts_trips_clean.csv`, `rts_tours.csv`

**Maryland Statewide Transportation Model (MSTM) 2016 skims, 1,588 SMZ/RMZ zones** — Maryland Department of Transportation (MDOT SHA), via the project's ABIT Maryland set-up (ABIT/input/mito_los/skims)
- Where: <https://www.roads.maryland.gov/ (verify: MSTM data are provided by MDOT on request; no download page is documented)>
- How: Obtain the MSTM 2016 skims in OMX format (matrix mat1, lookups origins / destinations): traveltime_auto.omx (AM-peak car time, min), distance_auto.omx (car distance, m), bus_tt.omx (bus time, min; 1000 = no path, 5 intrazonal), rail_tt00.omx (rail time, min; 0 = no rail path). convert_mstm_skims.py turns them into the csv.gz files the models read.
- Put in: `00_shared_data/raw/mstm_skims_2016`
- Files: `traveltime_auto.omx`, `distance_auto.omx`, `bus_tt.omx`, `rail_tt00.omx`

**MSTM zone system: SMZ/RMZ polygons, zone table, ABIT zone attributes** — MDOT (MSTM), via ABIT/input/mito_los/Zonal and 2_silo/input/zoneSystem.csv; ABIT/input/maryland/zone_attributes.csv (ABIT Maryland set-up)
- Where: <https://www.roads.maryland.gov/ (verify: provided with the MSTM)>
- How: Zone polygons SMZ_RMZ_02152011inMSTM_EPSG26918.shp (+ .dbf .shx .prj; field id / SMZRMZ, 1,588 zones), zoneSystem.csv (ZoneId, COUNTYFIPS, ...), zone_attributes.csv (id, centroid x/y EPSG:26985, countyfips, region types, bbsr, area_km2; built for ABIT from mstm_26985.shp and zoneSystem.csv, see 3_abit/coef_respec/PROVENANCE.md, script not in the project).
- Put in: `00_shared_data/raw/mstm_zone_system`
- Files: `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.dbf/.shx/.prj`, `zoneSystem.csv`, `zone_attributes.csv`

**Census 2010 Gazetteer file of census tracts (internal points)** — U.S. Census Bureau
- Where: <https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html (verify: the project copy I695_pricing/data/census/Gaz_tracts_national.txt has no download log)>
- How: 2010 Gazetteer, Census Tracts, national file Gaz_tracts_national.txt (GEOID, INTPTLAT, INTPTLONG). Used by survey_population.tract_zone: zone of each tract's internal point.
- Put in: `00_shared_data/raw/census_gazetteer_2010`
- Files: `Gaz_tracts_national.txt`

In [ ]:
raw = os.path.join(FOLDER, "input", "data", "raw")
if os.path.isdir(raw):
    display(nt.files_table(raw))
    for f in sorted(glob.glob(os.path.join(raw, "*")))[:6]:
        head, n = nt.preview(f)
        if head is not None:
            print(f"\n{os.path.basename(f)}" + (f"  ({n:,} rows)" if n else ""))
            display(head)
shared = os.path.join(FOLDER, "..", "00_shared_data")
print("shared data folders:", sorted(os.listdir(shared)))

## 2. Data preparation

Shared and upstream stages (outputs provided in `../00_shared_data` and `input/data/raw`; run them only for a full rebuild from the raw data; `cd` paths from this model folder):

- **Step 1.** shared preparation S00: Download / request the raw data (DATA_SOURCES.json) into 00_shared_data/raw/<folder> (one README per folder says what goes there)  
  `cd ../00_shared_prep/code` → `sh fetch_gtfs.sh && sh fetch_gtfs_regional.sh   # the GTFS feeds; the other sources are manual downloads or requests`
- **Step 2.** shared preparation S01: Survey cleaning: rts_trips_clean.csv and rts_tours.csv from the RTS flat file (01_clean_rts.py of the tour-based MITO project)  
  `cd ../00_shared_prep/code` → `none: the cleaning script is not on this machine; use the provided files`
- **Step 3.** shared preparation S02: Estimation / hold-out split of the survey households (seed 2026, stratified by county x household size)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python make_rts_split.py`
- **Step 4.** shared preparation S03: Tract internal point -> model zone (tract_zone.csv) and persons per zone in 2016 (zone_population_2016.csv)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 python3 build_survey_population.py --rebuild-zone-files   # also runs S04; needs geopandas (project Python)`
- **Step 5.** shared preparation S04: Survey population of both halves in the ABIT input format (households, persons, dwellings, person map, workplaces)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python build_survey_population.py   # uses the provided tract_zone.csv / zone_population_2016.csv; add --rebuild-zone-files for S03`
- **Step 6.** shared preparation S05: Typed survey episodes of both halves (TASHA activity types, person segments, zones of the trip ends)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python build_episodes.py`
- **Step 7.** shared preparation S06: MSTM 2016 skims OMX -> csv.gz (car time, car distance, MSTM bus and rail time)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python convert_mstm_skims.py`

Package steps, run below when `RUN_DATA_PREP = True`: they rebuild `input/data/estimation` and `input/data/holdout`; `check_against_provided.py` compares the rebuilt tables with the provided ones.

**Step 8.** optional (data already provided in input/data/estimation and input/data/holdout): rebuild the estimation tables from input/data/raw (about 0.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('PYTHONDONTWRITEBYTECODE=1 python3 build_estimation_tables.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 9.** input_check: compare your rebuilt estimation data with the provided data (row counts, columns, column sums, max abs difference; Biogeme parts by column statistics) (about 1 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('/Users/raassarkertomal/.venvs/biogeme/bin/python input/code/check_against_provided.py', cwd=os.path.join(FOLDER, ''))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

## 3. Estimation data

The tables exactly as used in the estimation, and the data dictionary.

In [ ]:
import estimate_outing_mode as M
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv")) if os.path.exists(os.path.join(est, "data_dictionary.csv")) else None
dd

In [ ]:
r = M.read("estimation")
df = r[0] if isinstance(r, tuple) else r
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 4. Descriptive statistics

In [ ]:
ycol = "y" if "y" in df else None
if ycol:
    c = df[ycol].value_counts().sort_index()
    t = c.rename("observations").to_frame(); t["share %"] = (100 * c / c.sum()).round(2)
    av = [x for x in df.columns if x.startswith("av_")]
    display(t)
    if av:
        display(df[av].mean().round(3).rename("share available").to_frame())
    ax = t["share %"].plot.bar(figsize=(5, 3), color="#2b6cb0", rot=0); ax.set_title("Chosen alternative (code y; see the data dictionary)"); plt.show()
wcol = next((w for w in ["wesml", "w", "weight"] if w in df), None)
if wcol:
    print(f"weight column {wcol}: mean {df[wcol].mean():.3f}, min {df[wcol].min():.3f}, max {df[wcol].max():.3f}")
X = nt.explanatory(df.drop(columns=[c for c in df.columns if c.startswith("av_") or c in ("y", "wesml", "w", "weight", "row_index")]))
display(X.describe().T.round(3))

## 5. Model specification

The folder's script `model/estimate_outing_mode.py`, imported as it is. Its utility functions:

In [ ]:
src = inspect.getsource(M)
print(src[:src.find("def read(")])

## 6. Correlation of the explanatory variables

|r| ≥ 0.7 and VIF ≥ 10 are listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
f = M.formulas() if hasattr(M, "formulas") else M.loglike()
expr = f["loglike"] if isinstance(f, dict) else f
used = [c for c in nt.model_variables(expr) if c in df and not c.startswith("av_") and c not in ("y", "wesml", "w", "weight")]
Xu = nt.explanatory(df[used])
print(f"{len(used)} data columns enter the utilities")
Cu = nt.corr_heatmap(Xu, "Variables of the utilities")
display(nt.high_pairs(Cu)); display(nt.vif(Xu))

## 7. Log-likelihood at the article coefficients

No estimation: the script's `evaluate("estimation", "article")`. Target: article -360.2,
Biogeme -360.1556 (N = 1177, K = 8).

In [ ]:
def evaluate(half, at):
    here = os.getcwd()
    try:
        return M.evaluate(half, at) if "at" in inspect.signature(M.evaluate).parameters else M.evaluate(half)
    finally:
        os.chdir(here)

if RUN_CHECK:
    t0 = time.time(); ll_article = evaluate("estimation", "article"); print(f"{time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

The script's `estimate()`: Biogeme with its defaults (`output/biogeme.toml`), robust standard errors. Report in
`output/outing_mode.html` and `.yaml`; table `output/estimated_parameters.csv`.

In [ ]:
if RUN_ESTIMATION:
    here = os.getcwd(); t0 = time.time()
    try:
        M.estimate(OUT)
    finally:
        os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s")
y = [f for f in sorted(glob.glob(os.path.join(OUT, "*.yaml")), key=os.path.getmtime) if "eval" not in os.path.basename(f)]
results = EstimationResults.from_yaml_file(y[-1]) if y else None
print(f"results read from {os.path.basename(y[-1])}" if y else "no result in output/: sections 9-10 are skipped")

## 9. Results

### Estimates in the article format
Coefficient (stars: *** p < 0.001, ** p < 0.01, * p < 0.05), robust s.e., p.

In [ ]:
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### Goodness of fit

In [ ]:
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "K": results.number_of_free_parameters, "N": 1177})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

Log-likelihood of the hold-out half at the full-precision estimates (`evaluate("holdout", "full")`).

In [ ]:
if RUN_HOLDOUT:
    ll_hold = evaluate("holdout", "full")
else:
    print("skipped (RUN_HOLDOUT = False)")